In [1]:
import practicalSPARQL
import pandas as pd
import ast
import matplotlib.pyplot as plt
import numpy as np
import bqplot as bq

In [2]:
root_q = 'queries'
root_d = 'results'

# read login data
login = pd.read_json('config_sparql.json')
ENDPOINT = login['endpoint'][0]
USERNAME = login['username'][0]
PASSWORD = login['password'][0]

# create sparql object
sparql = practicalSPARQL.practicalWrapper(ENDPOINT)
sparql.setCredentials(USERNAME, PASSWORD)

print("--- Querying ENDPOINT: {} ---".format(ENDPOINT))

--- Querying ENDPOINT: http://devmeta.sphaera.mpiwg-berlin.mpg.de/sparql ---


In [3]:
q = practicalSPARQL.stringify_SPARQL('elements_query_050824.sparql')    
# select data from the ttl file as a dataframe
df = sparql.select_as_dataframe(q)

In [4]:
q = practicalSPARQL.stringify_SPARQL('books_query.sparql')    # select data from the ttl file as a dataframe
books = sparql.select_as_dataframe(q)

In [5]:
# df

In [6]:
#adding year intervals (to both df, de and books, for future plotting)

# Ensure filtered_df and books are treated as copies (if required)
df = df.copy()
books = books.copy()

# Ensure the year columns are of integer type (use .loc to avoid SettingWithCopyWarning)
df.loc[:, 'year'] = df['year'].astype(int)
books.loc[:, 'year'] = books['year'].astype(int)

# Define bins and labels for year intervals
bins = [1470, 1490, 1510, 1530, 1550, 1570, 1590, 1610, 1630, 1650]
labels = [
    '1470-1489', '1490-1509', '1510-1529', '1530-1549',
    '1550-1569', '1570-1589', '1590-1609', '1610-1629', '1630-1650'
]

# Add 'interval' column to the filtered_df DataFrame based on custom bins (use .loc to avoid SettingWithCopyWarning)
df.loc[:, 'year_interval'] = pd.cut(df['year'], bins=bins, labels=labels, right=False)

# Add 'interval' column to the books DataFrame (use .loc to avoid SettingWithCopyWarning)
books.loc[:, 'year_interval'] = pd.cut(books['year'], bins=bins, labels=labels, right=False)

In [7]:
#adding place categories (to both df)

# Define the place categories
large_centers = ['Venice', 'Paris', 'Wittenberg']
medium_centers = ['Antwerp', 'Leipzig', 'Frankfurt (Main)', 'Lyon', 'Cologne', 'London']
small_centers = ['Rome', 'Strasbourg', 'Seville', 'Leiden', 'Milan', 'Saint Gervais', 
                 'Florence', 'Kraków', 'Salamanca', 'Lisbon', 'Bologna', 'Madrid', 
                 'Sine loco', 'Basel', 'Lemgo', 'Dijon', 'Valladolid', 'Perugia']
one_book_centers = ['Siena', 'Avignon', 'Vienna', 'Ferrara', 'Padua', 'Nuremberg', 
                    'Neustadt an der Weinstraße', 'Mexico City', 'Mainz', 'Coimbra', 'Leuven', 
                    'Ingolstadt', 'Heidelberg', 'Geneva', 'Dillingen an der Donau', 'Alcalá de Henares']

# Create a dictionary mapping cities to place categories
city_to_category = {}

for city in large_centers:
    city_to_category[city] = 'Large Center'
for city in medium_centers:
    city_to_category[city] = 'Medium Center'
for city in small_centers:
    city_to_category[city] = 'Small Center'
for city in one_book_centers:
    city_to_category[city] = 'One Book Center'

# Function to assign place category based on city name
def assign_place_category(city):
    return city_to_category.get(city, 'Unknown')  # Default to 'Unknown' if the city is not in the list

# Add the 'place_category' column to df
df['place_category'] = df['place'].apply(assign_place_category)

# Add the 'place_category' column to books
books['place_category'] = books['place'].apply(assign_place_category)

In [8]:
#add geographical data (longitude and latitude)

# Define the city_position dictionary (same as before)
city_position = {
    'Alcalá de Henares': (40.4818396, -3.3644973),
    'Antwerp': (51.2211097, 4.3997081),
    'Augsburg': (48.3690341, 10.8979522),
    'Avignon': (43.9492493, 4.8059012),
    'Basel': (47.5581077, 7.5878261),
    'Bologna': (44.4938203, 11.3426327),
    'Bordeaux': (44.841225, -0.5800364),
    'Coimbra': (40.2111931, -8.4294632),
    'Cologne': (43.7218277, 0.9774958),
    'Dijon': (47.3215806, 5.0414701),
    'Dillingen an der Donau': (48.5812791, 10.4951026),
    'Dortmund': (51.5142273, 7.4652789),
    'Ferrara': (44.8372737, 11.6186451),
    'Florence': (43.7697955, 11.2556404),
    'Frankfurt am Main': (50.1106444, 8.6820917),
    'Frankfurt an der Oder': (52.3412273, 14.549452),
    'Geneva': (46.2047169, 6.1423106290939335),
    'Heidelberg': (49.4093582, 8.694724),
    'Ingolstadt': (48.7630165, 11.4250395),
    'Kraków': (50.0469432, 19.997153435836697),
    'Leiden': (52.1594747, 4.4908843),
    'Leipzig': (51.3406321, 12.3747329),
    'Lemgo': (52.0280674, 8.9012894),
    'Leuven': (50.879202, 4.7011675),
    'Lisbon': (38.7077507, -9.1365919),
    'London': (51.4893335, -0.14405508452768728),
    'Lyon': (45.7578137, 4.8320114),
    'Madrid': (40.4167047, -3.7035825),
    'Mainz': (50.0012314, 8.2762513),
    'Mexico City': (19.4326296, -99.1331785),
    'Milan': (45.4641943, 9.1896346),
    'Neustadt an der Weinstraße': (49.3539802, 8.1350021),
    'Nuremberg': (49.453872, 11.077298),
    'Padua': (45.4077172, 11.8734455),
    'Paris': (48.8534951, 2.3483915),
    'Perugia': (43.1119613, 12.3890104),
    'Pesaro': (43.9098114, 12.9131228),
    'Rome': (41.8933203, 12.4829321),
    'Saint Gervais': (45.2022356, 5.4820229),
    'Salamanca': (40.9651572, -5.6640182),
    'Seville': (37.3886303, -5.9953403),
    'Siena': (43.3185536, 11.3316533),
    'Sine loco': None,
    'Strasbourg': (48.584614, 7.7507127),
    'Tournon': (45.0675156, 4.832852),
    'Tübingen': (48.5236164, 9.0535531),
    'Valladolid': (41.6521328, -4.728562),
    'Venice': (45.4371908, 12.3345898),
    'Vienna': (48.2083537, 16.3725042),
    'Wittenberg': (51.8666527, 12.646761)
}

# Assuming 'merged_df' is your existing DataFrame and has a 'place' column
# Create a function to map the place name to latitude and longitude
def get_coordinates(city):
    return city_position.get(city, (None, None))  # Returns None if city is not found in the dictionary

# Apply the function to create 'latitude' and 'longitude' columns
df[['latitude', 'longitude']] = df['place'].apply(lambda city: pd.Series(get_coordinates(city)))

# Display the updated DataFrame
#merged_df

# # Drop rows where 'latitude' or 'longitude' is NaN
# merged_df_clean = merged_df.dropna(subset=['latitude', 'longitude'])



In [9]:
df['cks'] = df['cks'].astype(str)
df['cks'] = df['cks'].str.split(', ')
df_exploded = df.explode('cks')

# Remove brackets, single quotes, double quotes, and leading/trailing whitespace
df_exploded['cks'] = df_exploded['cks'].str.replace(r"[\[\]\"']", "", regex=True).str.strip()

df_exploded.reset_index(drop=True, inplace=True)

In [10]:
df['cks'] = df['cks'].astype(str)
df['cks'] = df['cks'].str.split(', ')
df_exploded = df.explode('cks')

# Remove brackets, single quotes, double quotes, and leading/trailing whitespace
df_exploded['cks'] = df_exploded['cks'].str.replace(r"[\[\]\"']", "", regex=True).str.strip()

df_exploded.reset_index(drop=True, inplace=True)

In [11]:
# df_exploded

In [12]:
# Define general and subgroup keywords
general_ck = 'CK_Centrality of Earth'
subgroup_cks = [
    'CK_Negligible Dimensions of the Earth',
    'CK_Central Intersection of Eclipses',
    'CK_Visibility of Half of the Sky from the Central Earth'
]

# Get all clusters tagged with the general keyword
general_clusters = set(df[df['cks'] == general_ck]['cluster_name'].unique())

# Initialize set for missing clusters
clusters_to_fix = set()

# Collect all subgroup-tagged clusters missing the general tag
for ck in subgroup_cks:
    subgroup_clusters = set(df[df['cks'] == ck]['cluster_name'].unique())
    missing = subgroup_clusters - general_clusters
    clusters_to_fix.update(missing)

# Create new rows for missing general tags
# We'll assume that 'location_group' or other columns can be matched from existing rows for that cluster
new_rows = df[df['cluster_name'].isin(clusters_to_fix)].drop_duplicates('cluster_name').copy()
new_rows['cks'] = general_ck

# Append new rows to the original DataFrame
df = pd.concat([df, new_rows], ignore_index=True)

In [13]:
part_ids_with_types = pd.read_csv('part_ids_with_types.csv')

In [14]:
# Ensure both key columns are strings
df_exploded['custom_identifier'] = df_exploded['custom_identifier'].astype(str)
part_ids_with_types['part_id'] = part_ids_with_types['part_id'].astype(str)

# Merge on matching columns (custom_identifier ↔ part_id)
df_exploded = df_exploded.merge(
    part_ids_with_types,
    left_on='custom_identifier',
    right_on='part_id',
    how='left'  # keep all rows from df_exploded
)

# Drop the redundant column from the right DataFrame
df_exploded = df_exploded.drop(columns='part_id')

# Display the final result
df_exploded


,images,cluster_name,cks,book,bid,part_or_adaption,part_or_adaption_label,type_label,custom_identifier,place,year,flag,year_interval,place_category,latitude,longitude,part_type
0,http://dev.sphaera.mpiwg-berlin.mpg.de/contain...,SAC_SIL_00803,CK_Populated Earth,http://sphaera.mpiwg-berlin.mpg.de/id/item/0be...,1924,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Anonymous commentary (foeliciter inchoat),"Content, Annotated",322,Venice,1488,nan,1470-1489,Large Center,45.437191,12.33459,adaption_100
1,http://dev.sphaera.mpiwg-berlin.mpg.de/contain...,SAC_SIL_00803,CK_Lunar Eclipse,http://sphaera.mpiwg-berlin.mpg.de/id/item/0be...,1924,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Anonymous commentary (foeliciter inchoat),"Content, Annotated",322,Venice,1488,nan,1470-1489,Large Center,45.437191,12.33459,adaption_100
2,http://dev.sphaera.mpiwg-berlin.mpg.de/contain...,SAC_SIL_00664,CK_Circles of Equant Deferent Epicycle,http://sphaera.mpiwg-berlin.mpg.de/id/item/0be...,1924,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Anonymous commentary (foeliciter inchoat),"Content, Annotated",322,Venice,1488,nan,1470-1489,Large Center,45.437191,12.33459,adaption_100
3,http://dev.sphaera.mpiwg-berlin.mpg.de/contain...,SAC_SIL_01876,CK_Circles of Equant Deferent Epicycle,http://sphaera.mpiwg-berlin.mpg.de/id/item/0be...,1924,http://sphaera.mpiwg-berlin.mpg.de/id/part/1ea...,Theoricae novae planetarum of Peurbach,"Original Part, Content",104,Venice,1488,nan,1470-1489,Large Center,45.437191,12.33459,other
4,http://dev.sphaera.mpiwg-berlin.mpg.de/contain...,SAC_SIL_01876,CK_Lunar Nodes,http://sphaera.mpiwg-berlin.mpg.de/id/item/0be...,1924,http://sphaera.mpiwg-berlin.mpg.de/id/part/1ea...,Theoricae novae planetarum of Peurbach,"Original Part, Content",104,Venice,1488,nan,1470-1489,Large Center,45.437191,12.33459,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27976,http://www.researchspace.org/ontology/ImageReg...,SAC_SIL_00448,CK_Sphericity of the Heavens,http://sphaera.mpiwg-berlin.mpg.de/id/item/ba9...,2278,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Clavius's commentary on Sacrobosco's Sphere,"Annotated, Content",295,Venice,1601,nan,1590-1609,Large Center,45.437191,12.33459,adaption_100
27977,http://www.researchspace.org/ontology/ImageReg...,SAC_SIL_01782,CK_Sphericity of the Heavens,http://sphaera.mpiwg-berlin.mpg.de/id/item/ba9...,2278,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Clavius's commentary on Sacrobosco's Sphere,"Annotated, Content",295,Venice,1601,nan,1590-1609,Large Center,45.437191,12.33459,adaption_100
27978,http://www.researchspace.org/ontology/ImageReg...,SAC_SIL_01782,CK_Apparent Size of Stars,http://sphaera.mpiwg-berlin.mpg.de/id/item/ba9...,2278,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Clavius's commentary on Sacrobosco's Sphere,"Annotated, Content",295,Venice,1601,nan,1590-1609,Large Center,45.437191,12.33459,adaption_100
27979,http://www.researchspace.org/ontology/ImageReg...,SAC_SIL_01955,CK_Sphericity of the Heavens,http://sphaera.mpiwg-berlin.mpg.de/id/item/ba9...,2278,http://sphaera.mpiwg-berlin.mpg.de/id/adaption...,Clavius's commentary on Sacrobosco's Sphere,"Annotated, Content",295,Venice,1601,nan,1590-1609,Large Center,45.437191,12.33459,adaption_100


In [15]:
df_exploded.to_csv('full_image_data_feb_25.csv', index=False)

In [16]:
books.to_csv('full_book_data_feb_25.csv', index=False)